In [27]:
# ---- System imports----

import sys
import os
import json
import pandas as pd
from tqdm import tqdm

# --- Project root setup ---

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added to sys.path:")
print(project_root)

Project root added to sys.path:
d:\Infosys Internship\infosys-langgraph-email-assistant-group2


In [28]:
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY missing"
assert os.getenv("LANGCHAIN_API_KEY"), "LANGCHAIN_API_KEY missing"



In [29]:
import importlib.util
print(importlib.util.find_spec("src"))

ModuleSpec(name='src', loader=<_frozen_importlib_external.NamespaceLoader object at 0x000001A51E9251D0>, submodule_search_locations=_NamespacePath(['d:\\Infosys Internship\\infosys-langgraph-email-assistant-group2\\src']))


In [30]:
from src.classifier import triage_node
print(triage_node)

<function triage_node at 0x000001A52261C4A0>


In [31]:
from src.email_routes import react_agent
print(react_agent)

<function react_agent at 0x000001A52261CB80>


In [32]:
from src.app_factory import build_app

app = build_app() # graph is defined internally

In [33]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df["email_text"] = df["body"]

df.head()

,id,sender,subject,body,priority,triage_label,email_text
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,Reminder: The client meeting is scheduled at 1...
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,Your invoice of INR 25515.09 is due on 2025-12...
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,Reminder: The client meeting is scheduled at 1...
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,"Hello team, please find the attached weekly re..."
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,"Hello team, please find the attached weekly re..."


In [35]:
import time
from tqdm import tqdm

replies = []

for _, row in tqdm(df.head(5).iterrows(), total=5):
    email = row["email_text"]

    agent_out = app.invoke({
        "email": email,
        "user": email
    })

    # ✅ FINAL & CORRECT
    reply_text = agent_out["response"]

    replies.append(reply_text)
    time.sleep(2)

df.loc[:4, "reply"] = replies
df.head()

100%|██████████| 5/5 [00:39<00:00,  7.92s/it]


,id,sender,subject,body,priority,triage_label,email_text,reply
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,Reminder: The client meeting is scheduled at 1...,Subject: Re: Reminder: The client meeting is s...
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,Your invoice of INR 25515.09 is due on 2025-12...,Subject: Re: Your invoice of INR 25515.09 is d...
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,Reminder: The client meeting is scheduled at 1...,Subject: Re: Reminder: The client meeting is s...
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,"Hello team, please find the attached weekly re...",Subject: Re: Weekly Report and Action Items\n\...
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,"Hello team, please find the attached weekly re...",Subject: Re: Weekly report and action items fo...


In [37]:
# Define LLM-as-Judge

from langchain_google_genai import ChatGoogleGenerativeAI

judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

def judge_response(email_text, reply_text):
    prompt = f"""
You are an evaluation judge.

Evaluate the assistant reply based on:
1. Politeness
2. Helpfulness
3. Accuracy
4. Overall Quality

Score each from 1 (poor) to 5 (excellent).

Email:
{email_text}

Assistant Reply:
{reply_text}

Return ONLY valid JSON in this format:
{{
  "politeness": int,
  "helpfulness": int,
  "accuracy": int,
  "overall": int
}}
No explanation. No markdown.
"""
    return judge_llm.invoke(prompt).content.strip()

In [38]:
# Run Agent + Judge on all Emails

import time
from tqdm import tqdm

results = []

for _, row in tqdm(df.head(5).iterrows(), total=5):
    email_text = row["email_text"]
    reply_text = row["reply"]

    judge_raw = judge_response(email_text, reply_text)

    results.append({
        "body": email_text,
        "reply": reply_text,
        "judge_raw": judge_raw
    })

    time.sleep(2)

100%|██████████| 5/5 [00:29<00:00,  5.90s/it]


In [39]:
import json
import pandas as pd

parsed = []

for r in results:
    scores = json.loads(r["judge_raw"])

    parsed.append({
        "body": r["body"],
        "reply": r["reply"],
        "politeness": scores["politeness"],
        "helpfulness": scores["helpfulness"],
        "accuracy": scores["accuracy"],
        "overall": scores["overall"]
    })

final_df = pd.DataFrame(parsed)
final_df

,body,reply,politeness,helpfulness,accuracy,overall
0,Reminder: The client meeting is scheduled at 1...,Subject: Re: Reminder: The client meeting is s...,5,5,5,5
1,Your invoice of INR 25515.09 is due on 2025-12...,Subject: Re: Your invoice of INR 25515.09 is d...,5,5,5,5
2,Reminder: The client meeting is scheduled at 1...,Subject: Re: Reminder: The client meeting is s...,5,5,5,5
3,"Hello team, please find the attached weekly re...",Subject: Re: Weekly Report and Action Items\n\...,5,4,5,5
4,"Hello team, please find the attached weekly re...",Subject: Re: Weekly report and action items fo...,5,4,5,5


In [40]:
metrics = final_df[["politeness", "helpfulness", "accuracy", "overall"]].mean()
metrics

politeness     5.0
helpfulness    4.6
accuracy       5.0
overall        5.0
dtype: float64

In [41]:
final_df.to_csv("../data/milestone2_final_output.csv", index=False)
print("Saved: data/milestone2_final_output.csv")

Saved: data/milestone2_final_output.csv
